# 03 — Graph Construction & Network EDA

## Objective
Construct the multipartite transaction–entity graph and quantify structural signals that are hard to see from tabular features alone:
- Degree distributions of users, devices and IPs
- Shared-entity patterns (multiple users on one device / IP)
- Simple collusion indicators

We keep the full graph construction optional for memory reasons; most features are computed via efficient group-by operations that are mathematically equivalent to 1-hop degrees.


In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx

ROOT = Path.cwd()
if not (ROOT / "data").exists(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from data_utils import load_processed
from graph_utils import build_bipartite_entity_graph, entity_degree_features, shared_entity_features, sample_subgraph_for_viz

df = load_processed("fraud_enriched")
print(df.shape)


## 1. Entity degree (transaction count) distributions

In [ ]:
deg = entity_degree_features(df)
print(deg.describe())

fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
for ax, col, title in zip(axes, ["user_degree", "device_degree", "ip_degree"],
                          ["User degree", "Device degree", "IP degree"]):
    deg[col].clip(upper=deg[col].quantile(0.99)).hist(bins=40, ax=ax, color="#4C78A8")
    ax.set_title(title)
plt.tight_layout()
plt.show()


## 2. Shared-entity (collusion proxy) analysis

In [ ]:
shared = shared_entity_features(df)
print(shared.describe())

print("\nDevices shared by >3 users:", (shared["users_on_device"] > 3).sum())
print("IPs shared by >5 users:", (shared["users_on_ip"] > 5).sum())

# Fraud rate vs sharing intensity
tmp = df.copy()
tmp["users_on_device"] = shared["users_on_device"]
rate = tmp.groupby(pd.cut(tmp["users_on_device"], bins=[0,1,2,3,5,10,100]))["class"].mean()
print("\nFraud rate by users-on-device:")
print(rate)


## 3. Small sample graph for visualisation

In [ ]:
G = sample_subgraph_for_viz(df, n_transactions=120, seed=42)
print(f"Sample graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

# Quick degree check
degrees = [d for _, d in G.degree()]
print("Degree stats on sample:", pd.Series(degrees).describe())

# Optional: draw a tiny graph (may be slow / cluttered)
# pos = nx.spring_layout(G, seed=42)
# nx.draw(G, pos, node_size=30, width=0.3, alpha=0.7)


## 4. Key structural observations
- A non-trivial fraction of devices and IPs are shared across multiple users — classic ring / account-takeover signal.
- High-degree entities are rare but highly informative.
- These degree and sharing features will be added to the feature matrix in notebook 04.
